<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_2/FastText_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastText: эмбеддинги на основе символьных n-грамм

## Введение: почему FastText заслуживает отдельного разбора

Word2Vec и GloVe, которые мы подробно разобрали, дают плотные семантически насыщенные векторы слов. Но у них есть общее фундаментальное ограничение: они работают со **словами как атомарными единицами**. Каждое слово — это неделимый токен, которому соответствует один вектор. Из этого вытекают три проблемы.

**Проблема 1: OOV (out-of-vocabulary).** Если слово не встречалось в обучающем корпусе, у него нет вектора. В реальных задачах OOV-слова возникают постоянно: новые термины, имена, опечатки, морфологические вариации. Модель просто не может их обработать.

**Проблема 2: морфология.** В языках с богатой морфологией (русский, немецкий, турецкий, финский) одно слово может иметь десятки форм. Word2Vec и GloVe учат каждую форму отдельно, что требует огромных корпусов и всё равно оставляет редкие формы плохо обученными. При этом формы связаны: «кошка», «кошки», «кошке», «кошку» — это одно и то же слово в разных падежах. Их векторы должны быть близки, но Word2Vec этого не гарантирует.

**Проблема 3: опечатки и неологизмы.** Слово «кошкаа» (с опечаткой) или «кошк» (обрезок) не имеет вектора, хотя интуитивно понятно, что оно близко к «кошке».

FastText, предложенный Пётром Бояновским, Эдуардом Граве, Арманом Жулиным и Томашем Миколовым в 2016 году (те же авторы, что и Word2Vec), решает эти проблемы элегантным способом: **слово представляется как сумма векторов его символьных n-грамм**. Вместо того чтобы учить вектор для каждого слова целиком, мы учим векторы для символьных n-грамм и составляем из них вектор слова.

Это даёт три преимущества:

1. **OOV решается:** даже если слово новое, его символьные n-граммы, скорее всего, уже встречались в других словах, и мы можем составить вектор.
2. **Морфология учитывается:** формы одного слова имеют общие n-граммы, поэтому их векторы автоматически близки.
3. **Опечатки обрабатываются:** «кошкаа» имеет n-граммы, общие с «кошкой», поэтому вектор будет близок.

В этой лекции мы подробно разберём:

- что такое символьные n-граммы и как они строятся;
- как FastText представляет слово через n-граммы;
- как меняются формулы Word2Vec (CBOW и Skip-gram) при переходе к FastText;
- как FastText решает проблему OOV;
- как FastText работает с морфологией;
- как обучать FastText;
- в чём отличие FastText от Word2Vec и GloVe;
- какие у FastText ограничения и расширения.

Мы будем следовать той же структуре, что и в лекциях по Word2Vec и GloVe.

---

## 1. Символьные n-граммы

### 1.1 Определение

**Символьная n-грамма** — это последовательность из $n$ символов. Например, для слова «кошка»:

- 1-граммы (униграммы): к, о, ш, к, а;
- 2-граммы (биграммы): ко, ош, шк, ка;
- 3-граммы (триграммы): кош, ошк, шка;
- 4-граммы: кошк, ошка;
- 5-граммы: кошка.

**Тонкий момент:** в FastText используются n-граммы с **граничными символами**. К слову добавляются специальные символы `<` в начало и `>` в конец. Например, слово «кошка» превращается в `<кошка>`. Это позволяет различать n-граммы, которые встречаются в начале, середине и конце слова.

Пример с граничными символами для слова «кошка» при $n = 3$:

- `<ко`, `кош`, `ошк`, `шка`, `ка>`.

Без граничных символов мы бы имели `кош`, `ошк`, `шка` — и не могли бы отличить, например, «кошка» от «кошкаа» (где `<ко` и `а>` дают дополнительную информацию).

**Стандартный набор n-грамм:** в FastText обычно используются n-граммы от $n = 3$ до $n = 6$. Это означает, что для каждого слова генерируются все n-граммы длины 3, 4, 5, 6. Число n-грамм для одного слова примерно равно $4 \times |w|$, где $|w|$ — длина слова.

### 1.2 Пример

Рассмотрим слово «кошка» (5 символов). С граничными символами: `<кошка>` (7 символов).

**3-граммы:** `<ко`, `кош`, `ошк`, `шка`, `ка>` — 5 штук.

**4-граммы:** `<кош`, `кошк`, `ошка`, `шка>` — 4 штуки.

**5-граммы:** `<кошк`, `кошка`, `ошка>` — 3 штуки.

**6-граммы:** `<кошка`, `кошка>` — 2 штуки.

**Всего:** $5 + 4 + 3 + 2 = 14$ n-грамм.

В реальности FastText добавляет к этому ещё и само слово целиком (как специальную n-грамму). Итого 15 элементов.

### 1.3 Хеширование n-грамм

Проблема: число возможных n-грамм огромно. Для словаря из миллиона слов и алфавита из 100 символов число возможных 3-грамм — $100^3 = 10^6$, 6-грамм — $100^6 = 10^{12}$. Хранить вектор для каждой n-граммы невозможно.

**Решение: хеширование.** Все n-граммы отображаются в фиксированное число «ведёрок» (buckets) с помощью хеш-функции. Например, $B = 2 \times 10^6$ ведёрок. Каждая n-грамма получает индекс:

$$
\text{index}(g) = \text{hash}(g) \mod B.
$$

Вектор для n-граммы — это строка матрицы $Z \in \mathbb{R}^{B \times d}$, где $B$ — число ведёрок, $d$ — размерность эмбеддинга.

**Тонкий момент:** из-за хеширования возможны коллизии — разные n-граммы могут попасть в одно ведро. Но на практике коллизии редки, и их влияние невелико. Это стандартный компромисс между памятью и точностью.

**Пример:** n-грамма `кош` может получить индекс 12345, а n-грамма `шка` — индекс 67890. Векторы этих n-грамм — строки 12345 и 67890 матрицы $Z$.

### 1.4 Матрица n-грамм

Обозначим матрицу векторов n-грамм через $Z \in \mathbb{R}^{B \times d}$. Строка $k$ матрицы $Z$ — это вектор $z_k$ для n-граммы, попавшей в ведро $k$.

**Размерность:** $B \times d$. При $B = 2 \times 10^6$ и $d = 300$ это $6 \times 10^8$ параметров. Это меньше, чем $N \times d$ для словаря $N = 10^6$ (тоже $3 \times 10^8$), но сопоставимо. Главное преимущество — n-граммы общие для разных слов, поэтому они получают больше обновлений.

---

## 2. Представление слова через n-граммы

### 2.1 Формула

Пусть слово $w$ состоит из набора n-грамм $G(w)$. Например, для «кошки»:

$$
G(\text{кошка}) = \{\text{<ко}, \text{кош}, \text{ошк}, \text{шка}, \text{ка>}, \text{<кош}, \ldots, \text{кошка}, \ldots\}.
$$

Вектор слова $w$ определяется как **сумма** (или среднее) векторов его n-грамм:

$$
u_w = \sum_{g \in G(w)} z_g,
$$

где $z_g \in \mathbb{R}^d$ — вектор n-граммы $g$.

**Тонкий момент:** в оригинальной статье FastText используется **суммирование**, а не усреднение. Это означает, что вектор слова имеет большую норму, если у слова много n-грамм. Для длинных слов норма больше. На практике это не проблема, потому что нормировка происходит внутри модели (скалярное произведение).

**Альтернатива:** можно использовать среднее:

$$
u_w = \frac{1}{|G(w)|} \sum_{g \in G(w)} z_g.
$$

Это делает норму вектора независимой от длины слова. Но в FastText используется суммирование, потому что оно проще и работает лучше на практике.

### 2.2 Включение самого слова

В FastText к набору n-грамм добавляется **само слово** как отдельный элемент. Это означает, что даже если все n-граммы слова совпадают с n-граммами других слов, само слово имеет уникальный вектор. Формально:

$$
G(w) = \{\text{<ко}, \text{кош}, \ldots, \text{ка>}\} \cup \{w\}.
$$

Вектор слова $w$ тогда:

$$
u_w = z_w + \sum_{g \in G_{\text{ngram}}(w)} z_g,
$$

где $z_w$ — вектор самого слова (как в Word2Vec), а $z_g$ — векторы n-грамм.

**Зачем это нужно?** Без этого слова, состоящие из одинаковых n-грамм, но разные по смыслу, имели бы одинаковые векторы. Например, «кошка» и «кошак» имеют много общих n-грамм, но разные значения. Добавление самого слова позволяет различать их.

### 2.3 Пример

Для слова «кошка» (предположим, $d = 2$):

- Вектор слова: $z_{\text{кошка}} = (0.1, -0.2)$.
- Векторы n-грамм: $z_{\text{<ко}} = (0.05, 0.03)$, $z_{\text{кош}} = (0.02, 0.04)$, $\ldots$

Тогда:

$$
u_{\text{кошка}} = (0.1, -0.2) + (0.05, 0.03) + (0.02, 0.04) + \ldots
$$

Результат — вектор, который учитывает как уникальность слова, так и его морфологию.

---

## 3. FastText на основе Skip-gram

### 3.1 Постановка задачи

FastText использует ту же идею, что и Word2Vec: предсказание контекста. Мы разберём Skip-gram версию, потому что она чаще используется на практике.

Для каждой позиции $t$ в корпусе:

- Целевое слово: $w_t$.
- Контекстные слова: $w_{t-m}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+m}$.

**Задача:** для каждой пары (целевое, контекстное) максимизировать вероятность $P(w_O \mid w_I)$.

### 3.2 Вероятность

Вероятность контекстного слова $w_O$ при условии целевого $w_I$:

$$
P(w_O \mid w_I) = \frac{\exp(v_{w_O}^\top u_{w_I})}{\sum_{w \in V} \exp(v_w^\top u_{w_I})},
$$

где:

- $u_{w_I} = \sum_{g \in G(w_I)} z_g$ — вектор целевого слова (сумма n-грамм);
- $v_{w_O}$ — выходной вектор контекстного слова (без n-грамм, обычный вектор).

**Тонкий момент:** в FastText n-граммы используются только для **входных** векторов (целевых слов). Выходные векторы остаются обычными словами. Это потому, что контекстные слова — это наблюдаемые слова, и для них не нужно решать проблему OOV.

**Исключение:** в некоторых реализациях n-граммы используются и для выходных векторов. Но стандартная версия — только для входных.

### 3.3 Функция правдоподобия

Для всего корпуса:

$$
\mathcal{L} = \prod_{t=1}^{T} \prod_{-m \le j \le m, j \ne 0} P(w_{t+j} \mid w_t).
$$

Логарифм:

$$
\ell = \sum_{t=1}^{T} \sum_{-m \le j \le m, j \ne 0} \log P(w_{t+j} \mid w_t).
$$

Подставляя выражение для $P$:

$$
\ell = \sum_{t=1}^{T} \sum_{-m \le j \le m, j \ne 0} \left[ v_{w_{t+j}}^\top u_{w_t} - \log \sum_{w \in V} \exp(v_w^\top u_{w_t}) \right].
$$

**Цель:** максимизировать $\ell$ по $z_g$ (n-граммы), $v_w$ (выходные векторы) и $z_w$ (сами слова).

### 3.4 Negative sampling для FastText

Как и в Word2Vec, знаменатель softmax требует суммирования по всему словарю. Решение — negative sampling.

Для каждой пары $(w_I, w_O)$ с $K$ отрицательными примерами:

$$
\mathcal{L}_{\text{neg}} = \log \sigma(v_{w_O}^\top u_{w_I}) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I}),
$$

где $u_{w_I} = \sum_{g \in G(w_I)} z_g$.

**Отрицательные примеры** выбираются из шумового распределения $P_n(w) \propto \text{count}(w)^{3/4}$, как в Word2Vec.

### 3.5 Градиенты

Выведем градиенты для одной пары $(w_I, w_O)$ с одним отрицательным примером $w_{\text{neg}}$.

**Обозначения:**

- $u = u_{w_I} = \sum_{g \in G(w_I)} z_g$ — вектор целевого слова;
- $v = v_{w_O}$ — выходной вектор контекстного слова;
- $v' = v_{w_{\text{neg}}}$ — выходной вектор отрицательного слова;
- $x = v^\top u$, $x' = v'^\top u$.

**Функция потерь:**

$$
\mathcal{L} = \log \sigma(x) + \log \sigma(-x').
$$

**Градиент по $v$:**

$$
\frac{\partial \mathcal{L}}{\partial v} = (1 - \sigma(x)) \cdot u.
$$

**Градиент по $v'$:**

$$
\frac{\partial \mathcal{L}}{\partial v'} = -\sigma(x') \cdot u.
$$

**Градиент по $u$:**

$$
\frac{\partial \mathcal{L}}{\partial u} = (1 - \sigma(x)) v - \sigma(x') v'.
$$

**Градиенты по $z_g$ (n-граммам целевого слова):**

Поскольку $u = \sum_{g \in G(w_I)} z_g$, градиент по каждой n-грамме:

$$
\frac{\partial \mathcal{L}}{\partial z_g} = \frac{\partial \mathcal{L}}{\partial u} = (1 - \sigma(x)) v - \sigma(x') v', \quad \forall g \in G(w_I).
$$

**Тонкий момент:** все n-граммы целевого слова получают **одинаковое** обновление. Это следствие суммирования. В отличие от Word2Vec, где вектор слова обновляется целиком, в FastText обновление распределяется между всеми n-граммами слова.

**Градиент по $z_{w_I}$ (самому слову):**

$$
\frac{\partial \mathcal{L}}{\partial z_{w_I}} = (1 - \sigma(x)) v - \sigma(x') v'.
$$

Это то же самое, что и для n-грамм, потому что слово тоже входит в сумму.

### 3.6 Обновление параметров

Используя градиентный подъём с $\eta$:

$$
v \leftarrow v + \eta (1 - \sigma(x)) u,
$$

$$
v' \leftarrow v' - \eta \sigma(x') u,
$$

$$
z_g \leftarrow z_g + \eta \left[ (1 - \sigma(x)) v - \sigma(x') v' \right], \quad \forall g \in G(w_I).
$$

$$
z_{w_I} \leftarrow z_{w_I} + \eta \left[ (1 - \sigma(x)) v - \sigma(x') v' \right].
$$

### 3.7 Полный алгоритм

1. **Инициализация:** случайные малые значения для $Z$ (n-граммы) и $V$ (слова).
2. **Для каждой эпохи:**
   - Для каждой позиции $t$:
     - Для каждого $j \in \{-m, \ldots, -1, 1, \ldots, m\}$:
       - $w_I = w_t$, $w_O = w_{t+j}$.
       - Вычислить $u_{w_I} = \sum_{g \in G(w_I)} z_g$.
       - Выбрать $K$ отрицательных примеров.
       - Вычислить $x = v_{w_O}^\top u_{w_I}$, $x'_k = v_{w_{\text{neg}_k}}^\top u_{w_I}$.
       - Обновить параметры.
3. **Повторять** до сходимости.

### 3.8 Вычислительная сложность

Одно обновление для пары $(w_I, w_O)$ требует:

- Вычисления $u_{w_I}$: $|G(w_I)|$ сложений векторов длины $d$. $|G(w_I)| \approx 4 |w_I|$ (от 3- до 6-грамм).
- Вычисления скалярных произведений: $K+1$ штук, каждое $O(d)$.
- Обновления: $|G(w_I)| + 1$ векторов n-грамм, $v$, $v'$, $K$ отрицательных.

**Сложность:** $O((K + |G(w_I)|) \cdot d)$. Поскольку $|G(w_I)|$ пропорционально длине слова, FastText медленнее Word2Vec примерно в $4$–$5$ раз на одно обновление. Но это компенсируется лучшим качеством для редких слов.

---

## 4. FastText на основе CBOW

### 4.1 Постановка задачи

В CBOW версии FastText контекстные слова усредняются, а целевое слово предсказывается.

Для позиции $t$:

- Контекст: $C_t = \{w_{t-m}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+m}\}$.
- Целевое слово: $w_t$.

**Задача:** максимизировать $P(w_t \mid C_t)$.

### 4.2 Представление контекста

Каждое контекстное слово $c \in C_t$ представляется через n-граммы:

$$
u_c = \sum_{g \in G(c)} z_g.
$$

Усреднённый вектор контекста:

$$
h = \frac{1}{2m} \sum_{c \in C_t} u_c = \frac{1}{2m} \sum_{c \in C_t} \sum_{g \in G(c)} z_g.
$$

**Тонкий момент:** каждое контекстное слово вносит вклад через свои n-граммы. Это означает, что даже если слово $c$ редкое, его n-граммы, скорее всего, общие с другими словами, и вклад будет осмысленным.

### 4.3 Вероятность

$$
P(w_t \mid C_t) = \frac{\exp(v_{w_t}^\top h)}{\sum_{w \in V} \exp(v_w^\top h)}.
$$

### 4.4 Negative sampling

$$
\mathcal{L} = \log \sigma(v_{w_t}^\top h) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top h).
$$

### 4.5 Градиенты

**Градиент по $v_{w_t}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{w_t}} = (1 - \sigma(x)) h.
$$

**Градиент по $v_{w_{\text{neg}}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') h.
$$

**Градиент по $h$:**

$$
\frac{\partial \mathcal{L}}{\partial h} = (1 - \sigma(x)) v_{w_t} - \sigma(x') v_{w_{\text{neg}}}.
$$

**Градиенты по $z_g$ (n-граммам контекстных слов):**

$$
\frac{\partial \mathcal{L}}{\partial z_g} = \frac{1}{2m} \frac{\partial \mathcal{L}}{\partial h}, \quad \forall g \in \bigcup_{c \in C_t} G(c).
$$

**Тонкий момент:** все n-граммы всех контекстных слов получают одинаковое обновление, делённое на $2m$. Это следствие усреднения.

### 4.6 Обновление

$$
z_g \leftarrow z_g + \frac{\eta}{2m} \left[ (1 - \sigma(x)) v_{w_t} - \sigma(x') v_{w_{\text{neg}}} \right].
$$

---

## 5. Как FastText решает проблему OOV

### 5.1 Проблема

Word2Vec и GloVe не могут обработать слово, которого не было в обучающем корпусе. У такого слова нет ни индекса в словаре, ни вектора.

FastText решает эту проблему потому, что вектор слова — это **сумма векторов его n-грамм**. Даже если слово новое, его n-граммы, скорее всего, уже встречались в других словах.

### 5.2 Пример

Пусть слово «кошкаа» (с опечаткой) не встречалось в корпусе. Но n-граммы `<ко`, `кош`, `ошк`, `шка`, `каа`, `аа>` и т.д. встречались в других словах. Вектор нового слова:

$$
u_{\text{кошкаа}} = \sum_{g \in G(\text{кошкаа})} z_g.
$$

Этот вектор будет близок к вектору «кошки», потому что большинство n-грамм общие.

### 5.3 Оценка качества OOV

FastText обычно показывает значительно лучшее качество на OOV-словах, чем Word2Vec и GloVe. В оригинальной статье показано, что на задаче аналогий для OOV-слов FastText даёт точность 70–80%, тогда как Word2Vec — около 0%.

**Тонкий момент:** качество OOV зависит от того, насколько n-граммы нового слова пересекаются с n-граммами известных слов. Для совершенно новых слов (например, «Xyzzy») качество будет низким, потому что их n-граммы тоже новые.

### 5.4 Хранение n-грамм

FastText хранит векторы только для **уникальных** n-грамм. Число уникальных n-грамм обычно в 3–10 раз больше числа слов. Для словаря $N = 10^6$ это $3 \times 10^6$–$10^7$ n-грамм. Хеширование позволяет ограничить память: $B = 2 \times 10^6$ ведёрок.

---

## 6. Как FastText работает с морфологией

### 6.1 Проблема морфологии

В языках с богатой морфологией (русский, немецкий, турецкий) одно слово может иметь десятки форм. Word2Vec учит каждую форму отдельно:

- «кошка» — вектор 1;
- «кошки» — вектор 2;
- «кошке» — вектор 3;
- и т.д.

Проблема: редкие формы плохо обучены, потому что встречаются редко. Но они связаны с частыми формами.

### 6.2 Решение FastText

FastText учит n-граммы, общие для всех форм:

- «кошка»: `<ко`, `кош`, `ошк`, `шка`, `ка>`, ...
- «кошки»: `<ко`, `кош`, `ошк`, `шки`, `ки>`, ...
- «кошке»: `<ко`, `кош`, `ошк`, `шке`, `ке>`, ...

Общие n-граммы (`<ко`, `кош`, `ошк`) получают много обновлений и хорошо обучаются. Это автоматически делает векторы форм близкими.

### 6.3 Пример

После обучения:

- $u_{\text{кошка}} \approx u_{\text{кошки}} \approx u_{\text{кошке}}$, потому что у них много общих n-грамм.

Это именно то, что нужно: формы одного слова семантически близки.

### 6.4 Аффиксы и приставки

FastText автоматически улавливает аффиксы и приставки. Например, в русском:

- «бежать», «бегу», «бежит» — общие n-граммы `<бе`, `беж`, `ежа`, ...
- «прибежать», «убежать» — общие n-граммы с «бежать» плюс уникальные приставки.

Это позволяет FastText обобщать на новые слова с знакомыми аффиксами.

---

## 7. Сравнение FastText с Word2Vec и GloVe

### 7.1 Общие черты

- Все три метода дают плотные векторы размерности $d = 100$–$300$.
- Все три используют скалярное произведение как меру совместимости.
- Все три используют negative sampling или его аналоги.
- Все три дают похожие результаты на стандартных бенчмарках для слов, которые были в корпусе.

### 7.2 Различия

| Свойство | Word2Vec | GloVe | FastText |
|----------|----------|-------|----------|
| Единица | Слово | Слово | Слово + n-граммы |
| OOV | Нет | Нет | Да |
| Морфология | Слабо | Слабо | Хорошо |
| Опечатки | Нет | Нет | Да |
| Память | $N \times d$ | $N \times N$ + $N \times d$ | $B \times d$ |
| Скорость | Быстро | Средне | Медленнее |
| Качество (in-vocab) | Хорошо | Хорошо | Хорошо |
| Качество (OOV) | Плохо | Плохо | Хорошо |

### 7.3 Когда что использовать

- **Word2Vec:** большие корпуса, все слова в словаре, ограниченное время.
- **GloVe:** средние корпуса, важна глобальная статистика.
- **FastText:** морфологически богатые языки, OOV-слова, опечатки, небольшие корпуса.

**Тонкий момент:** на практике FastText часто даёт лучшее качество, чем Word2Vec и GloVe, даже для in-vocab слов, потому что n-граммы действуют как регуляризация.

---

## 8. Практические детали

### 8.1 Гиперпараметры

- $d = 100$–$300$: размерность эмбеддинга.
- $n_{\min} = 3$, $n_{\max} = 6$: диапазон длин n-грамм.
- $B = 2 \times 10^6$: число ведёрок для хеширования.
- $m = 5$: размер окна.
- $K = 5$–$10$: число отрицательных примеров.
- $\eta = 0.05$: начальная скорость обучения.
- Эпох: 5–10.

### 8.2 Инициализация

Векторы n-грамм инициализируются случайными малыми значениями. Слово тоже имеет свой вектор (как в Word2Vec).

### 8.3 Субсэмплирование

Как и в Word2Vec, применяется субсэмплирование частых слов:

$$
P_{\text{discard}}(w) = 1 - \sqrt{\frac{t}{f(w)}}, \quad t = 10^{-4}.
$$

### 8.4 Динамическое окно

Размер окна выбирается случайно от 1 до $m$ для каждой пары.

### 8.5 AdaGrad

FastText использует AdaGrad для адаптации скорости обучения. Это особенно важно для n-грамм, потому что их частоты сильно различаются.

### 8.6 Нормализация

После обучения векторы нормализуют по L2.

### 8.7 Оценка качества

- **In-vocab:** аналогии, близость слов.
- **OOV:** аналогии для OOV-слов, downstream-задачи.
- **Морфология:** задачи, требующие учёта форм слов.

---

## 9. Расширения FastText

### 9.1 FastText для классификации

Помимо эмбеддингов, FastText имеет **режим классификации**. В этом режиме модель предсказывает метки для документов. Архитектура:

- Вход: усреднённый вектор n-грамм документа.
- Выход: вероятности меток через softmax или иерархический softmax.

Это очень быстрый и эффективный метод для классификации текстов. Он показывает качество, сравнимое с нейросетями, но обучается за секунды.

### 9.2 FastText для предложений и документов

Усреднение векторов слов документа (с n-граммами) даёт хороший baseline для эмбеддингов документов.

### 9.3 Многоязычный FastText

FastText можно обучать на нескольких языках одновременно, что позволяет получать сопоставимые векторы для переводов.

### 9.4 FastText с субсловными единицами

Помимо символьных n-грамм, можно использовать слоги (например, BPE — byte pair encoding). Это используется в современных моделях (BERT, GPT).

---

## 10. Ограничения FastText

### 10.1 Полисемия

Как и Word2Vec и GloVe, FastText даёт один вектор на слово, независимо от контекста. Слово «банк» (финансовый и речной) получает один вектор.

### 10.2 Вычислительная сложность

FastText медленнее Word2Vec в 4–5 раз на одно обновление, потому что нужно вычислять и обновлять векторы n-грамм.

### 10.3 Память

Хотя хеширование ограничивает память, FastText всё равно требует больше памяти, чем Word2Vec, потому что нужно хранить векторы n-грамм.

### 10.4 Гиперпараметры

FastText имеет больше гиперпараметров ($n_{\min}$, $n_{\max}$, $B$), чем Word2Vec. Их выбор влияет на качество.

### 10.5 Коллизии хеширования

Из-за хеширования возможны коллизии: разные n-граммы могут попасть в одно ведро и получить одинаковый вектор. Это может ухудшить качество.

---

## 11. Заключение

FastText — это элегантное расширение Word2Vec, которое решает три фундаментальные проблемы: OOV, морфологию и опечатки. Ключевая идея: **слово — это сумма векторов его символьных n-грамм**.

**Ключевые формулы:**

Вектор слова:

$$
u_w = z_w + \sum_{g \in G_{\text{ngram}}(w)} z_g.
$$

Вероятность контекстного слова (Skip-gram):

$$
P(w_O \mid w_I) = \frac{\exp(v_{w_O}^\top u_{w_I})}{\sum_{w \in V} \exp(v_w^\top u_{w_I})}.
$$

Функция потерь с negative sampling:

$$
\mathcal{L} = \log \sigma(v_{w_O}^\top u_{w_I}) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I}).
$$

Градиенты:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_O}} = (1 - \sigma(x)) u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial z_g} = (1 - \sigma(x)) v_{w_O} - \sigma(x') v_{w_{\text{neg}}}, \quad \forall g \in G(w_I).
$$

FastText — это важный шаг в эволюции методов эмбеддингов. Он показывает, что работа с подсловными единицами позволяет решать проблемы, которые казались фундаментальными. Его идеи (n-граммы, хеширование, сумма векторов) используются в современных моделях, включая BERT (WordPiece) и GPT (BPE).

---

**В следующей части** мы разберём **численный пример FastText** на нашем учебном корпусе: построение n-грамм, вычисление векторов слов, один шаг обучения и сравнение с Word2Vec.

# Численный пример FastText на учебном корпусе

## 1. Постановка задачи

Рассмотрим тот же учебный корпус из трёх документов:

- $d_1$: «кошка сидит на окне»
- $d_2$: «собака сидит на крыльце»
- $d_3$: «кошка спит на диване»

Объединим документы в одну последовательность:

$$
\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{сидит}, \text{на}, \text{крыльце}, \text{кошка}, \text{спит}, \text{на}, \text{диване}.
$$

Длина последовательности $T = 12$. Словарь:

$$
V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\},
$$

размер словаря $N = 8$.

**Параметры:**

- окно $m = 1$;
- размерность эмбеддинга $d = 2$;
- $K = 1$ отрицательный пример;
- скорость обучения $\eta = 0.1$;
- n-граммы: $n = 3$ (только триграммы, для простоты).

**Тонкий момент:** в реальных задачах $d = 100$–$300$, $m = 5$–$10$, $K = 5$–$20$, $n_{\min} = 3$, $n_{\max} = 6$. Мы используем маленькие значения, чтобы вычисления были обозримыми.

## 2. Генерация символьных n-грамм

### 2.1 Формирование n-грамм

Для каждого слова $w$ добавим граничные символы `<` и `>`: слово $w$ превращается в `<w>`. Затем извлечём все 3-граммы.

**Слово «сидит»:** `<сидит>` (7 символов).

3-граммы: `<си`, `сид`, `иди`, `дит`, `ит>`.

Плюс само слово: `сидит`.

Итого: 6 элементов.

**Слово «кошка»:** `<кошка>` (7 символов).

3-граммы: `<ко`, `кош`, `ошк`, `шка`, `ка>`.

Плюс само слово: `кошка`.

Итого: 6 элементов.

**Слово «на»:** `<на>` (4 символа).

3-граммы: `<на`, `на>`.

Плюс само слово: `на`.

Итого: 3 элемента.

**Слово «диване»:** `<диване>` (8 символов).

3-граммы: `<ди`, `див`, `ива`, `ван`, `ане`, `не>`.

Плюс само слово: `диване`.

Итого: 7 элементов.

**Слово «собака»:** `<собака>` (8 символов).

3-граммы: `<со`, `соб`, `оба`, `бак`, `ака`, `ка>`.

Плюс само слово: `собака`.

Итого: 7 элементов.

**Слово «окне»:** `<окне>` (6 символов).

3-граммы: `<ок`, `окн`, `кне`, `не>`.

Плюс само слово: `окне`.

Итого: 5 элементов.

**Слово «крыльце»:** `<крыльце>` (9 символов).

3-граммы: `<кр`, `кры`, `рыл`, `ыль`, `льц`, `ьце`, `це>`.

Плюс само слово: `крыльце`.

Итого: 8 элементов.

**Слово «спит»:** `<спит>` (6 символов).

3-граммы: `<сп`, `спи`, `пит`, `ит>`.

Плюс само слово: `спит`.

Итого: 5 элементов.

### 2.2 Уникальные n-граммы

Объединим все n-граммы и уберём дубликаты. Вот полный список уникальных 3-грамм:

| № | n-грамма | Из каких слов |
|---|----------|---------------|
| 1 | `<ко` | кошка |
| 2 | `кош` | кошка |
| 3 | `ошк` | кошка |
| 4 | `шка` | кошка |
| 5 | `ка>` | кошка, собака |
| 6 | `<си` | сидит |
| 7 | `сид` | сидит |
| 8 | `иди` | сидит |
| 9 | `дит` | сидит |
| 10 | `ит>` | сидит, спит |
| 11 | `<на` | на |
| 12 | `на>` | на |
| 13 | `<ок` | окне |
| 14 | `окн` | окне |
| 15 | `кне` | окне |
| 16 | `не>` | окне, диване |
| 17 | `<со` | собака |
| 18 | `соб` | собака |
| 19 | `оба` | собака |
| 20 | `бак` | собака |
| 21 | `ака` | собака |
| 22 | `<кр` | крыльце |
| 23 | `кры` | крыльце |
| 24 | `рыл` | крыльце |
| 25 | `ыль` | крыльце |
| 26 | `льц` | крыльце |
| 27 | `ьце` | крыльце |
| 28 | `це>` | крыльце |
| 29 | `<сп` | спит |
| 30 | `спи` | спит |
| 31 | `пит` | спит |
| 32 | `<ди` | диване |
| 33 | `див` | диване |
| 34 | `ива` | диване |
| 35 | `ван` | диване |
| 36 | `ане` | диване |

Итого: **36 уникальных 3-грамм**. В реальных задачах используется хеширование, чтобы ограничить память, но для нашего примера мы будем хранить вектор для каждой уникальной n-граммы.

## 3. Инициализация параметров

### 3.1 Векторы слов (входные)

Эти векторы используются как $z_w$ — вектор самого слова, который добавляется к сумме n-грамм.

| Слово | $z_w$ |
|-------|-------|
| кошка | $(0.2, -0.1)$ |
| сидит | $(0.3, 0.4)$ |
| на | $(-0.1, 0.6)$ |
| окне | $(0.5, -0.3)$ |
| собака | $(0.1, 0.2)$ |
| крыльце | $(-0.4, 0.1)$ |
| спит | $(0.6, 0.5)$ |
| диване | $(-0.2, -0.5)$ |

### 3.2 Векторы слов (выходные)

Эти векторы используются для контекстных слов и отрицательных примеров.

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.1, 0.3)$ |
| сидит | $(-0.2, 0.4)$ |
| на | $(0.5, -0.1)$ |
| окне | $(0.3, 0.2)$ |
| собака | $(-0.3, -0.2)$ |
| крыльце | $(0.4, 0.5)$ |
| спит | $(-0.1, 0.6)$ |
| диване | $(0.2, -0.4)$ |

### 3.3 Векторы n-грамм

Для нашего примера нам понадобятся векторы n-грамм для слов «сидит», «кошка» и «на» (потому что мы будем обрабатывать пары с этими словами). Зададим их малыми случайными значениями.

**N-граммы слова «сидит»:**

| n-грамма | $z_g$ |
|----------|-------|
| `<си` | $(0.01, 0.02)$ |
| `сид` | $(0.02, -0.01)$ |
| `иди` | $(-0.01, 0.03)$ |
| `дит` | $(0.03, 0.01)$ |
| `ит>` | $(0.01, -0.02)$ |

**N-граммы слова «кошка»:**

| n-грамма | $z_g$ |
|----------|-------|
| `<ко` | $(0.02, 0.01)$ |
| `кош` | $(-0.01, 0.02)$ |
| `ошк` | $(0.01, -0.01)$ |
| `шка` | $(0.02, 0.03)$ |
| `ка>` | $(-0.02, 0.01)$ |

**N-граммы слова «на»:**

| n-грамма | $z_g$ |
|----------|-------|
| `<на` | $(0.01, -0.01)$ |
| `на>` | $(0.02, 0.02)$ |

**N-граммы слова «диване»:**

| n-грамма | $z_g$ |
|----------|-------|
| `<ди` | $(0.01, 0.03)$ |
| `див` | $(-0.02, 0.01)$ |
| `ива` | $(0.02, -0.02)$ |
| `ван` | $(0.01, 0.02)$ |
| `ане` | $(-0.01, 0.01)$ |
| `не>` | $(0.02, -0.01)$ |

**N-граммы слова «собака»:**

| n-грамма | $z_g$ |
|----------|-------|
| `<со` | $(0.02, 0.01)$ |
| `соб` | $(-0.01, 0.02)$ |
| `оба` | $(0.01, -0.02)$ |
| `бак` | $(0.02, 0.01)$ |
| `ака` | $(-0.02, 0.02)$ |
| `ка>` | $(-0.02, 0.01)$ |

**Тонкий момент:** n-грамма `ка>` встречается и в «кошке», и в «собаке». В FastText это одна и та же n-грамма с одним вектором. Это ключевое свойство: общие n-граммы разделяются между словами.

## 4. Первая пара: (сидит, кошка)

Возьмём позицию $t = 2$: целевое слово «сидит», контекстное слово «кошка». Отрицательный пример — «диване».

### 4.1 Вычисление вектора целевого слова

Вектор целевого слова «сидит» — это сумма вектора самого слова и векторов его n-грамм:

$$
u_{\text{сидит}} = z_{\text{сидит}} + z_{\text{<си}} + z_{\text{сид}} + z_{\text{иди}} + z_{\text{дит}} + z_{\text{ит>}}.
$$

Вычислим покомпонентно:

Первая компонента:

$$
0.3 + 0.01 + 0.02 + (-0.01) + 0.03 + 0.01 = 0.36.
$$

Вторая компонента:

$$
0.4 + 0.02 + (-0.01) + 0.03 + 0.01 + (-0.02) = 0.43.
$$

Итак:

$$
u_{\text{сидит}} = (0.36, 0.43).
$$

### 4.2 Прямой проход

**Шаг 1: скалярное произведение для положительной пары.**

$$
v_{\text{кошка}} = (0.1, 0.3).
$$

$$
x = v_{\text{кошка}}^\top u_{\text{сидит}} = 0.1 \cdot 0.36 + 0.3 \cdot 0.43 = 0.036 + 0.129 = 0.165.
$$

**Шаг 2: сигмоида.**

$$
\sigma(x) = \sigma(0.165) = \frac{1}{1 + e^{-0.165}} = \frac{1}{1 + 0.8479} = \frac{1}{1.8479} \approx 0.5412.
$$

**Шаг 3: скалярное произведение для отрицательной пары.**

$$
v_{\text{диване}} = (0.2, -0.4).
$$

$$
x' = v_{\text{диване}}^\top u_{\text{сидит}} = 0.2 \cdot 0.36 + (-0.4) \cdot 0.43 = 0.072 - 0.172 = -0.100.
$$

**Шаг 4: сигмоида.**

$$
\sigma(x') = \sigma(-0.100) = \frac{1}{1 + e^{0.100}} = \frac{1}{1 + 1.1052} = \frac{1}{2.1052} \approx 0.4750.
$$

**Шаг 5: функция потерь.**

$$
\mathcal{L} = \log \sigma(x) + \log \sigma(-x') = \log(0.5412) + \log(0.5250).
$$

$$
\log(0.5412) \approx -0.6140, \quad \log(0.5250) \approx -0.6444.
$$

$$
\mathcal{L} \approx -0.6140 - 0.6444 = -1.2584.
$$

### 4.3 Обратный проход

**Градиент по $v_{\text{кошка}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{кошка}}} = (1 - \sigma(x)) \cdot u_{\text{сидит}} = (1 - 0.5412) \cdot (0.36, 0.43) = 0.4588 \cdot (0.36, 0.43) = (0.1652, 0.1973).
$$

**Градиент по $v_{\text{диване}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{диване}}} = -\sigma(x') \cdot u_{\text{сидит}} = -0.4750 \cdot (0.36, 0.43) = (-0.1710, -0.2043).
$$

**Градиент по $u_{\text{сидит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = (1 - \sigma(x)) \cdot v_{\text{кошка}} - \sigma(x') \cdot v_{\text{диване}}.
$$

Первое слагаемое:

$$
0.4588 \cdot (0.1, 0.3) = (0.0459, 0.1376).
$$

Второе слагаемое:

$$
0.4750 \cdot (0.2, -0.4) = (0.0950, -0.1900).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = (0.0459, 0.1376) - (0.0950, -0.1900) = (-0.0491, 0.3276).
$$

### 4.4 Обновление параметров

**Обновляем $v_{\text{кошка}}$:**

$$
v_{\text{кошка}} \leftarrow (0.1, 0.3) + 0.1 \cdot (0.1652, 0.1973) = (0.1 + 0.0165, 0.3 + 0.0197) = (0.1165, 0.3197).
$$

**Обновляем $v_{\text{диване}}$:**

$$
v_{\text{диване}} \leftarrow (0.2, -0.4) + 0.1 \cdot (-0.1710, -0.2043) = (0.2 - 0.0171, -0.4 - 0.0204) = (0.1829, -0.4204).
$$

**Обновляем вектор слова $z_{\text{сидит}}$:**

$$
z_{\text{сидит}} \leftarrow (0.3, 0.4) + 0.1 \cdot (-0.0491, 0.3276) = (0.3 - 0.0049, 0.4 + 0.0328) = (0.2951, 0.4328).
$$

**Обновляем векторы n-грамм слова «сидит»:**

Все n-граммы получают **одинаковое** обновление, потому что градиент по $u_{\text{сидит}}$ распределяется между ними равномерно.

$$
\frac{\partial \mathcal{L}}{\partial z_g} = \frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = (-0.0491, 0.3276), \quad \forall g \in G(\text{сидит}).
$$

Обновление для каждой n-граммы:

$$
z_g \leftarrow z_g + 0.1 \cdot (-0.0491, 0.3276) = z_g + (-0.0049, 0.0328).
$$

| n-грамма | Было | Стало |
|----------|------|-------|
| `<си` | $(0.01, 0.02)$ | $(0.0051, 0.0528)$ |
| `сид` | $(0.02, -0.01)$ | $(0.0151, 0.0228)$ |
| `иди` | $(-0.01, 0.03)$ | $(-0.0149, 0.0628)$ |
| `дит` | $(0.03, 0.01)$ | $(0.0251, 0.0428)$ |
| `ит>` | $(0.01, -0.02)$ | $(0.0051, 0.0128)$ |

**Проверка:** новый вектор $u_{\text{сидит}}$:

$$
u_{\text{сидит}} = (0.2951, 0.4328) + (0.0051, 0.0528) + (0.0151, 0.0228) + (-0.0149, 0.0628) + (0.0251, 0.0428) + (0.0051, 0.0128).
$$

Первая компонента:

$$
0.2951 + 0.0051 + 0.0151 - 0.0149 + 0.0251 + 0.0051 = 0.3306.
$$

Вторая компонента:

$$
0.4328 + 0.0528 + 0.0228 + 0.0628 + 0.0428 + 0.0128 = 0.6268.
$$

$$
u_{\text{сидит}} = (0.3306, 0.6268).
$$

**Наблюдение:** вектор целевого слова изменился значительно: с $(0.36, 0.43)$ до $(0.3306, 0.6268)$. Это произошло потому, что все 6 компонент (слово + 5 n-грамм) получили обновление.

## 5. Вторая пара: (сидит, на)

Та же позиция $t = 2$, но второе контекстное слово — «на». Отрицательный пример — «собака».

**Важно:** используем **обновлённый** вектор $u_{\text{сидит}} = (0.3306, 0.6268)$.

### 5.1 Прямой проход

$$
v_{\text{на}} = (0.5, -0.1).
$$

$$
x = v_{\text{на}}^\top u_{\text{сидит}} = 0.5 \cdot 0.3306 + (-0.1) \cdot 0.6268 = 0.1653 - 0.0627 = 0.1026.
$$

$$
\sigma(x) = \sigma(0.1026) = \frac{1}{1 + e^{-0.1026}} = \frac{1}{1 + 0.9025} = \frac{1}{1.9025} \approx 0.5256.
$$

Отрицательный пример:

$$
v_{\text{собака}} = (-0.3, -0.2).
$$

$$
x' = v_{\text{собака}}^\top u_{\text{сидит}} = (-0.3) \cdot 0.3306 + (-0.2) \cdot 0.6268 = -0.0992 - 0.1254 = -0.2246.
$$

$$
\sigma(x') = \sigma(-0.2246) = \frac{1}{1 + e^{0.2246}} = \frac{1}{1 + 1.2519} = \frac{1}{2.2519} \approx 0.4441.
$$

Функция потерь:

$$
\mathcal{L} = \log(0.5256) + \log(0.5559) \approx -0.6432 - 0.5874 = -1.2306.
$$

### 5.2 Обратный проход

**Градиент по $v_{\text{на}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{на}}} = (1 - 0.5256) \cdot (0.3306, 0.6268) = 0.4744 \cdot (0.3306, 0.6268) = (0.1568, 0.2973).
$$

**Градиент по $v_{\text{собака}}$:**

$$
\frac{\partial \mathcal{L}}{\partial v_{\text{собака}}} = -0.4441 \cdot (0.3306, 0.6268) = (-0.1468, -0.2784).
$$

**Градиент по $u_{\text{сидит}}$:**

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = 0.4744 \cdot (0.5, -0.1) - 0.4441 \cdot (-0.3, -0.2).
$$

Первое слагаемое:

$$
0.4744 \cdot (0.5, -0.1) = (0.2372, -0.0474).
$$

Второе слагаемое:

$$
0.4441 \cdot (-0.3, -0.2) = (-0.1332, -0.0888).
$$

Вычитаем:

$$
\frac{\partial \mathcal{L}}{\partial u_{\text{сидит}}} = (0.2372, -0.0474) - (-0.1332, -0.0888) = (0.3704, 0.0414).
$$

### 5.3 Обновление параметров

**Обновляем $v_{\text{на}}$:**

$$
v_{\text{на}} \leftarrow (0.5, -0.1) + 0.1 \cdot (0.1568, 0.2973) = (0.5 + 0.0157, -0.1 + 0.0297) = (0.5157, -0.0703).
$$

**Обновляем $v_{\text{собака}}$:**

$$
v_{\text{собака}} \leftarrow (-0.3, -0.2) + 0.1 \cdot (-0.1468, -0.2784) = (-0.3 - 0.0147, -0.2 - 0.0278) = (-0.3147, -0.2278).
$$

**Обновляем $z_{\text{сидит}}$ и n-граммы:**

$$
z_{\text{сидит}} \leftarrow (0.2951, 0.4328) + 0.1 \cdot (0.3704, 0.0414) = (0.2951 + 0.0370, 0.4328 + 0.0041) = (0.3321, 0.4369).
$$

Все n-граммы получают то же обновление:

| n-грамма | Было | Стало |
|----------|------|-------|
| `<си` | $(0.0051, 0.0528)$ | $(0.0421, 0.0569)$ |
| `сид` | $(0.0151, 0.0228)$ | $(0.0521, 0.0269)$ |
| `иди` | $(-0.0149, 0.0628)$ | $(0.0221, 0.0669)$ |
| `дит` | $(0.0251, 0.0428)$ | $(0.0621, 0.0469)$ |
| `ит>` | $(0.0051, 0.0128)$ | $(0.0421, 0.0169)$ |

**Проверка:** новый вектор $u_{\text{сидит}}$:

$$
u_{\text{сидит}} = (0.3321, 0.4369) + (0.0421, 0.0569) + (0.0521, 0.0269) + (0.0221, 0.0669) + (0.0621, 0.0469) + (0.0421, 0.0169).
$$

Первая компонента:

$$
0.3321 + 0.0421 + 0.0521 + 0.0221 + 0.0621 + 0.0421 = 0.5526.
$$

Вторая компонента:

$$
0.4369 + 0.0569 + 0.0269 + 0.0669 + 0.0469 + 0.0169 = 0.6514.
$$

$$
u_{\text{сидит}} = (0.5526, 0.6514).
$$

## 6. Сводка обновлений после двух пар

| Параметр | Начальное | После пары 1 | После пары 2 |
|----------|-----------|--------------|--------------|
| $u_{\text{сидит}}$ | $(0.36, 0.43)$ | $(0.3306, 0.6268)$ | $(0.5526, 0.6514)$ |
| $v_{\text{кошка}}$ | $(0.1, 0.3)$ | $(0.1165, 0.3197)$ | $(0.1165, 0.3197)$ |
| $v_{\text{на}}$ | $(0.5, -0.1)$ | $(0.5, -0.1)$ | $(0.5157, -0.0703)$ |
| $v_{\text{диване}}$ | $(0.2, -0.4)$ | $(0.1829, -0.4204)$ | $(0.1829, -0.4204)$ |
| $v_{\text{собака}}$ | $(-0.3, -0.2)$ | $(-0.3, -0.2)$ | $(-0.3147, -0.2278)$ |
| $z_{\text{сидит}}$ | $(0.3, 0.4)$ | $(0.2951, 0.4328)$ | $(0.3321, 0.4369)$ |
| $z_{\text{<си}}$ | $(0.01, 0.02)$ | $(0.0051, 0.0528)$ | $(0.0421, 0.0569)$ |
| $z_{\text{сид}}$ | $(0.02, -0.01)$ | $(0.0151, 0.0228)$ | $(0.0521, 0.0269)$ |
| $z_{\text{иди}}$ | $(-0.01, 0.03)$ | $(-0.0149, 0.0628)$ | $(0.0221, 0.0669)$ |
| $z_{\text{дит}}$ | $(0.03, 0.01)$ | $(0.0251, 0.0428)$ | $(0.0621, 0.0469)$ |
| $z_{\text{ит>}}$ | $(0.01, -0.02)$ | $(0.0051, 0.0128)$ | $(0.0421, 0.0169)$ |

**Наблюдения:**

- Вектор $u_{\text{сидит}}$ изменился дважды, потому что «сидит» было целевым словом в обеих парах.
- N-граммы слова «сидит» получили два обновления, потому что они входят в $u_{\text{сидит}}$.
- Выходные векторы $v_{\text{кошка}}$, $v_{\text{на}}$, $v_{\text{диване}}$, $v_{\text{собака}}$ обновились в своих парах.
- Все n-граммы слова «сидит» обновляются **одинаково**, потому что градиент по $u_{\text{сидит}}$ распределяется между ними равномерно.

## 7. Проверка: как изменилась вероятность

Рассмотрим, как изменилась вероятность $P(D=1 \mid \text{сидит}, \text{кошка})$ после обновления.

**До обучения:**

$$
x = v_{\text{кошка}}^\top u_{\text{сидит}} = 0.1 \cdot 0.36 + 0.3 \cdot 0.43 = 0.165.
$$

$$
\sigma(x) = 0.5412.
$$

**После обновления (пары 1):**

$$
u_{\text{сидит}} = (0.3306, 0.6268), \quad v_{\text{кошка}} = (0.1165, 0.3197).
$$

$$
x_{\text{new}} = 0.1165 \cdot 0.3306 + 0.3197 \cdot 0.6268 = 0.0385 + 0.2004 = 0.2389.
$$

$$
\sigma(x_{\text{new}}) = \sigma(0.2389) = \frac{1}{1 + e^{-0.2389}} = \frac{1}{1 + 0.7876} = \frac{1}{1.7876} \approx 0.5594.
$$

**Наблюдение:** вероятность выросла с 0.5412 до 0.5594. Это означает, что модель стала более уверена в реальности пары (сидит, кошка). Разница небольшая, потому что мы сделали только один шаг с маленькой скоростью обучения.

## 8. Как FastText использует общие n-граммы

Рассмотрим слово «собака». Его n-грамма `ка>` совпадает с n-граммой слова «кошка». Это означает, что вектор $z_{\text{ка>}}$ получает обновления от **обоих** слов.

**Пример:** если бы мы обработали пару (собака, сидит), то n-грамма `ка>` получила бы обновление. Это обновление повлияло бы и на вектор слова «кошка», потому что «кошка» тоже использует `ка>`.

**Интерпретация:** общие n-граммы действуют как **мост** между словами. Они позволяют модели обобщать знания с одних слов на другие. Если «кошка» и «собака» имеют общие n-граммы, их векторы будут ближе, чем если бы они не имели общих n-грамм.

**Проверка:** после обучения (несколько эпох) векторы «кошки» и «собаки» будут близки, потому что:

- они имеют общие n-граммы (`ка>`);
- они встречаются в похожих контекстах («сидит»).

Это ключевое преимущество FastText перед Word2Vec: даже если слово редкое, его n-граммы могут быть частыми, и вектор будет осмысленным.

## 9. Итоговые эмбеддинги (после нескольких эпох)

После нескольких эпох (проходов по всему корпусу) параметры стабилизируются. Приведём примерные итоговые значения (округлённо до 2 знаков).

**Входные векторы слов $z_w$:**

| Слово | $z_w$ |
|-------|-------|
| кошка | $(0.25, -0.12)$ |
| сидит | $(0.35, 0.45)$ |
| на | $(-0.08, 0.58)$ |
| окне | $(0.48, -0.28)$ |
| собака | $(0.15, 0.18)$ |
| крыльце | $(-0.38, 0.08)$ |
| спит | $(0.55, 0.48)$ |
| диване | $(-0.18, -0.48)$ |

**Векторы n-грамм (примеры):**

| n-грамма | $z_g$ |
|----------|-------|
| `<ко` | $(0.05, 0.03)$ |
| `кош` | $(0.02, 0.04)$ |
| `ошк` | $(0.03, -0.02)$ |
| `шка` | $(0.04, 0.05)$ |
| `ка>` | $(0.01, 0.02)$ |
| `<си` | $(0.06, 0.04)$ |
| `сид` | $(0.03, 0.02)$ |
| `иди` | $(0.02, 0.05)$ |
| `дит` | $(0.04, 0.03)$ |
| `ит>` | $(0.03, 0.01)$ |

**Итоговые векторы слов $u_w$ (сумма слова и n-грамм):**

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.25+0.05+0.02+0.03+0.04+0.01, -0.12+0.03+0.04-0.02+0.05+0.02) = (0.40, 0.00)$ |
| сидит | $(0.35+0.06+0.03+0.02+0.04+0.03, 0.45+0.04+0.02+0.05+0.03+0.01) = (0.53, 0.60)$ |
| собака | $(0.15+0.02-0.01+0.01+0.02-0.02+0.01, 0.18+0.01+0.02-0.02+0.01+0.02+0.02) = (0.18, 0.24)$ |

**Наблюдение:** векторы «кошки» $(0.40, 0.00)$ и «собаки» $(0.18, 0.24)$ не очень близки в этом примере, потому что мы обработали только 2 пары. После полного обучения (все пары, несколько эпох) они станут ближе, потому что имеют общие n-граммы (`ка>`) и встречаются с «сидит».

## 10. Сравнение с Word2Vec на том же корпусе

Проведём сравнение FastText и Word2Vec (Skip-gram) на одном и том же корпусе.

**Число обновлений за эпоху:**

- Word2Vec: 23 пары.
- FastText: 23 пары (те же пары, но каждое обновление затрагивает больше параметров).

**Число параметров:**

- Word2Vec: $N \times d + N \times d = 8 \times 2 + 8 \times 2 = 32$.
- FastText: $N \times d + N \times d + B \times d$, где $B$ — число уникальных n-грамм. В нашем случае $B = 36$, так что $8 \times 2 + 8 \times 2 + 36 \times 2 = 16 + 16 + 72 = 104$.

**Скорость:**

- Word2Vec: одно обновление требует $O(d)$ операций.
- FastText: одно обновление требует $O(|G(w)| \cdot d)$ операций, где $|G(w)|$ — число n-грамм слова. В нашем случае $|G(\text{сидит})| = 5$, так что FastText медленнее примерно в 5 раз.

**Качество:**

- Word2Vec: хорошее для in-vocab слов, плохое для OOV.
- FastText: хорошее для in-vocab и OOV слов, лучше для морфологически богатых языков.

**Ключевое отличие:** FastText обновляет не только вектор слова, но и векторы n-грамм. Это означает, что даже редкое слово получает много обновлений через свои n-граммы.

## 11. Заключение

В этом численном примере мы шаг за шагом вычислили FastText для учебного корпуса. Основные выводы:

1. **Слово представляется как сумма вектора самого слова и векторов его символьных n-грамм.** Это ключевая идея FastText.

2. **N-граммы разделяются между словами.** Общие n-граммы (`ка>`, `ит>`, `не>`) получают обновления от нескольких слов, что позволяет модели обобщать.

3. **Градиент по вектору слова распределяется между всеми n-граммами.** Это означает, что все n-граммы слова обновляются одинаково.

4. **FastText решает проблему OOV.** Даже если слово новое, его n-граммы, скорее всего, уже встречались в других словах.

5. **FastText учитывает морфологию.** Формы одного слова имеют общие n-граммы, поэтому их векторы автоматически близки.

6. **FastText медленнее Word2Vec,** потому что нужно вычислять и обновлять векторы n-грамм. Но это компенсируется лучшим качеством для редких слов и OOV.

**Ключевые формулы:**

Вектор слова:

$$
u_w = z_w + \sum_{g \in G(w)} z_g.
$$

Вероятность пары (Skip-gram):

$$
P(D=1 \mid w_I, w_O) = \sigma(v_{w_O}^\top u_{w_I}).
$$

Функция потерь:

$$
\mathcal{L} = \log \sigma(v_{w_O}^\top u_{w_I}) + \sum_{k=1}^{K} \log \sigma(-v_{w_{\text{neg}_k}}^\top u_{w_I}).
$$

Градиенты:

$$
\frac{\partial \mathcal{L}}{\partial v_{w_O}} = (1 - \sigma(x)) u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial v_{w_{\text{neg}}}} = -\sigma(x') u_{w_I},
$$

$$
\frac{\partial \mathcal{L}}{\partial z_g} = (1 - \sigma(x)) v_{w_O} - \sigma(x') v_{w_{\text{neg}}}, \quad \forall g \in G(w_I).
$$

Эти формулы — основа FastText. Их понимание позволяет эффективно обучать эмбеддинги для морфологически богатых языков и обрабатывать OOV-слова.